# CARE-BED — final offline analysis: A13 + B12, W20_HSG, 25-fold LORO

This notebook performs the final **leave-one-recording-out (LORO)** analysis for the configuration selected by the
window-size ablation.

Final dataset:

```text
Environment A: 13 historical recordings
Environment B: 12 recordings = s01, s02, s03 from each of 4 activities
TOTAL:         25 independent recordings
```

Selected configuration:

```text
W20_HSG
20-frame input
Hampel filter
Savitzky-Golay filter
54 retained amplitude features
BiLSTM(64)
stride = 5
```

## Evaluation protocol

```text
25 independent recordings
→ 25 folds
→ exactly 1 complete recording is TEST in each fold
→ remaining 24 recordings are DEVELOPMENT
```

Within every development recording:

```text
first 80% raw frames  → TRAIN
last 20% raw frames   → VALIDATION
```

The held-out recording is never used for:

- training,
- validation,
- fitting the StandardScaler,
- early stopping.

Preprocessing order:

```text
raw recording partition
→ Hampel
→ Savitzky-Golay
→ remove 10 subcarriers
→ 20-frame windows, stride 5
→ StandardScaler fitted only on TRAIN windows
→ BiLSTM
```

The main result is calculated by concatenating the held-out predictions from **all 25 folds**.

This notebook is **evaluation only**. After this analysis, train the final single model separately on all 25 recordings
and evaluate that deployment model on a fresh live sequence.

## 0. Install packages

In [ ]:
!pip -q install hampel joblib

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
from math import sqrt
import csv
import json
import random
import re
import shutil
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hampel import hampel
from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Optional Google Drive checkpointing

Because 25 folds can take a while, this notebook can save each completed fold to Google Drive.

Leave:

```python
USE_GOOGLE_DRIVE = True
```

if you are running in Colab. If the runtime disconnects, rerunning the notebook will reuse completed fold files.

If you do not want Drive access, set it to `False`.

In [ ]:
USE_GOOGLE_DRIVE = True

RUNTIME_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

if USE_GOOGLE_DRIVE and Path("/content").exists():
    try:
        from google.colab import drive

        drive.mount(
            "/content/drive",
            force_remount=False,
        )

        SAVE_DIR = Path(
            "/content/drive/MyDrive/CARE_BED_W20_25fold_LORO"
        )

    except Exception as exc:
        print(
            "Google Drive mount unavailable; using runtime storage instead:",
            exc,
        )

        SAVE_DIR = RUNTIME_DIR / "CARE_BED_W20_25fold_LORO"

else:
    SAVE_DIR = RUNTIME_DIR / "CARE_BED_W20_25fold_LORO"

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Result/checkpoint directory:", SAVE_DIR)

## 3. Upload the two dataset ZIPs

Upload:

```text
envA_dataset.zip
carebed_new_data.zip
```

In [ ]:
WORKDIR = RUNTIME_DIR

OLD_ZIP = WORKDIR / "envA_dataset.zip"
NEW_ZIP = WORKDIR / "carebed_new_data.zip"

if not OLD_ZIP.exists() or not NEW_ZIP.exists():
    try:
        from google.colab import files

        print(
            "Upload envA_dataset.zip and carebed_new_data.zip"
        )

        files.upload()

    except ImportError:
        pass

if not OLD_ZIP.exists():
    raise FileNotFoundError(
        "Missing envA_dataset.zip"
    )

if not NEW_ZIP.exists():
    raise FileNotFoundError(
        "Missing carebed_new_data.zip"
    )

OLD_ROOT = WORKDIR / "envA_dataset"
NEW_ROOT = WORKDIR / "carebed_new_data"

for root in [
    OLD_ROOT,
    NEW_ROOT,
]:
    if root.exists():
        shutil.rmtree(
            root
        )

with zipfile.ZipFile(
    OLD_ZIP,
    "r",
) as zf:
    zf.extractall(
        WORKDIR
    )

with zipfile.ZipFile(
    NEW_ZIP,
    "r",
) as zf:
    zf.extractall(
        WORKDIR
    )

if not OLD_ROOT.exists():
    raise FileNotFoundError(
        "envA_dataset.zip must contain top-level envA_dataset/"
    )

if not NEW_ROOT.exists():
    raise FileNotFoundError(
        "carebed_new_data.zip must contain top-level carebed_new_data/"
    )

print("Datasets extracted.")

## 4. Configuration

In [ ]:
EXPECTED_ROLE = "AP"
NEW_MAC = "24:0A:C4:02:29:F0"

OLD_SOURCE_TOKENS = {
    "inactivity": "br_ak",
    "lying down": "lezenie",
    "sitting up": "siedzenie",
    "fidgeting": "wiercenie",
}

# Historical Environment A boundaries: 1-based and inclusive.
OLD_RECORDING_RANGES = {
    "inactivity": [
        ("A_inactivity_s01", 1, 1765),
        ("A_inactivity_s02", 1766, 2427),
        ("A_inactivity_s03", 2428, 3095),
        ("A_inactivity_s04", 3096, 3202),
    ],
    "lying down": [
        ("A_lying_down_s01", 1, 607),
        ("A_lying_down_s02", 608, 1779),
        ("A_lying_down_s03", 1780, 2947),
    ],
    "sitting up": [
        ("A_sitting_up_s01", 1, 601),
        ("A_sitting_up_s02", 602, 1739),
        ("A_sitting_up_s03", 1740, 2922),
    ],
    "fidgeting": [
        ("A_fidgeting_s01", 1, 1123),
        ("A_fidgeting_s02", 1124, 1810),
        ("A_fidgeting_s03", 1811, 2955),
    ],
}

EXPECTED_OLD_LINE_COUNTS = {
    "inactivity": 3202,
    "lying down": 2947,
    "sitting up": 2922,
    "fidgeting": 2955,
}

B_FOLDERS = {
    "inactivity": "inactivity",
    "lying down": "lying_down",
    "sitting up": "sitting_up",
    "fidgeting": "fidgeting",
}

# Final predefined Environment B subset.
VALID_B_SESSIONS = {
    1,
    2,
    3,
}

FIRST_CSI_VALUES = 128
RAW_FEATURES = 64

COLUMNS_TO_DROP = [
    2,
    3,
    4,
    5,
    32,
    59,
    60,
    61,
    62,
    63,
]

N_FEATURES = 54

WINDOW_LENGTH = 20
STRIDE = 5

DEV_TRAIN_FRACTION = 0.80

HAMPEL_WINDOW = 10
SAVGOL_WINDOW = 10
SAVGOL_POLYORDER = 3

LEARNING_RATE = 0.0005
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 5

CACHE_FILE = SAVE_DIR / "carebed_A13_B12_W20_HSG_processed_cache.joblib"

DATASET_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_dataset_summary.csv"
FOLD_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_metrics.csv"
PREDICTIONS_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_predictions.csv"
REPORT_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_classification_report.csv"
ENVIRONMENT_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_environment_metrics.csv"
PER_RECORDING_CSV = SAVE_DIR / "carebed_A13_B12_W20_25fold_per_recording.csv"

CONFUSION_PDF = SAVE_DIR / "carebed_A13_B12_W20_25fold_confusion_matrix.pdf"
PER_RECORDING_PDF = SAVE_DIR / "carebed_A13_B12_W20_25fold_per_recording_accuracy.pdf"
LOSS_PDF = SAVE_DIR / "carebed_A13_B12_W20_25fold_learning_loss.pdf"
ACCURACY_PDF = SAVE_DIR / "carebed_A13_B12_W20_25fold_learning_accuracy.pdf"

CHECKPOINT_DIR = SAVE_DIR / "fold_checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_ZIP = SAVE_DIR / "CARE_BED_FINAL_A13_B12_W20_HSG_25fold_LORO_results.zip"

assert (
    RAW_FEATURES
    - len(
        COLUMNS_TO_DROP
    )
    == N_FEATURES
)

## 5. CSI parsing helpers

In [ ]:
def parse_bracket_payload(line):
    try:
        content = re.findall(
            r"\[(.*)\]",
            line,
        )[0]

        return [
            int(x)
            for x in content.split(" ")
            if x != ""
        ]

    except Exception:
        return None


def amplitude_from_payload(
    payload
):
    if (
        payload is None
        or len(
            payload
        ) < FIRST_CSI_VALUES
    ):
        return None

    raw = payload[
        :FIRST_CSI_VALUES
    ]

    imaginary = raw[
        0::2
    ]

    real = raw[
        1::2
    ]

    if len(
        real
    ) != RAW_FEATURES:
        return None

    return np.asarray(
        [
            sqrt(
                imaginary[i] ** 2
                + real[i] ** 2
            )
            for i in range(
                RAW_FEATURES
            )
        ],
        dtype=np.float32,
    )


def old_line_to_amplitude(
    line
):
    return amplitude_from_payload(
        parse_bracket_payload(
            line
        )
    )


def new_line_to_amplitude(
    line
):
    try:
        row = next(
            csv.reader(
                [
                    line
                ]
            )
        )

    except Exception:
        return None

    if len(
        row
    ) != 26:
        return None

    if row[
        0
    ].strip() != "CSI_DATA":
        return None

    if row[
        1
    ].strip() != EXPECTED_ROLE:
        return None

    if row[
        2
    ].strip().upper() != NEW_MAC.upper():
        return None

    try:
        declared_len = int(
            row[
                24
            ].strip()
        )

    except ValueError:
        return None

    payload = parse_bracket_payload(
        line
    )

    if (
        payload is None
        or len(
            payload
        ) != declared_len
    ):
        return None

    return amplitude_from_payload(
        payload
    )


def session_number_from_name(
    path
):
    match = re.search(
        r"_s(\d+)\.csv$",
        path.name,
        flags=re.IGNORECASE,
    )

    return (
        int(
            match.group(
                1
            )
        )
        if match
        else None
    )

## 6. Load Environment A: 13 historical recordings

In [ ]:
old_source_files = {}

for activity, token in OLD_SOURCE_TOKENS.items():
    matches = [
        path
        for path in OLD_ROOT.glob(
            "*.csv"
        )
        if token
        in path.name.lower()
    ]

    if len(
        matches
    ) != 1:
        raise FileNotFoundError(
            f"Expected one Environment A CSV containing '{token}', "
            f"found {[p.name for p in matches]}"
        )

    old_source_files[
        activity
    ] = matches[
        0
    ]


recordings_A = []

for activity, ranges in OLD_RECORDING_RANGES.items():
    source = old_source_files[
        activity
    ]

    with source.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as fh:
        lines = fh.readlines()

    expected = EXPECTED_OLD_LINE_COUNTS[
        activity
    ]

    if len(
        lines
    ) != expected:
        raise ValueError(
            f"{source.name}: expected {expected} lines, found {len(lines)}"
        )

    for recording_id, start1, end1 in ranges:
        amplitudes = [
            old_line_to_amplitude(
                line
            )
            for line in lines[
                start1 - 1:end1
            ]
        ]

        amplitudes = [
            value
            for value in amplitudes
            if value is not None
        ]

        raw = pd.DataFrame(
            amplitudes
        )

        if raw.shape[
            1
        ] != RAW_FEATURES:
            raise ValueError(
                f"{recording_id}: expected 64 features, got {raw.shape}"
            )

        recordings_A.append({
            "recording_id": recording_id,
            "label": activity,
            "environment": "Environment A",
            "source_file": source.name,
            "raw": raw.reset_index(
                drop=True
            ),
        })


summary_A = pd.DataFrame([
    {
        "recording_id": r[
            "recording_id"
        ],
        "label": r[
            "label"
        ],
        "frames": len(
            r[
                "raw"
            ]
        ),
    }
    for r in recordings_A
])

display(
    summary_A
)

print(
    "Environment A recordings:",
    len(
        recordings_A
    ),
)

print(
    "Environment A frames:",
    summary_A[
        "frames"
    ].sum(),
)

assert len(
    recordings_A
) == 13

assert summary_A[
    "frames"
].sum() == 12026

## 7. Load Environment B: only s01, s02 and s03

In [ ]:
recordings_B = []

for activity, folder_name in B_FOLDERS.items():
    folder = NEW_ROOT / folder_name

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing Environment B folder: {folder}"
        )

    for path in sorted(
        folder.glob(
            "*.csv"
        )
    ):
        if path.name.endswith(
            "_timing.csv"
        ):
            continue

        session_number = session_number_from_name(
            path
        )

        if session_number not in VALID_B_SESSIONS:
            continue

        amplitudes = []

        with path.open(
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as fh:
            for line in fh:
                amplitude = new_line_to_amplitude(
                    line
                )

                if amplitude is not None:
                    amplitudes.append(
                        amplitude
                    )

        if not amplitudes:
            raise ValueError(
                f"{path.name}: no valid CSI frames parsed."
            )

        raw = pd.DataFrame(
            amplitudes
        )

        if raw.shape[
            1
        ] != RAW_FEATURES:
            raise ValueError(
                f"{path.name}: expected 64 features, got {raw.shape}"
            )

        recordings_B.append({
            "recording_id": (
                "B_"
                + path.stem.replace(
                    "envb_",
                    "",
                )
            ),
            "label": activity,
            "environment": "Environment B",
            "source_file": path.name,
            "raw": raw.reset_index(
                drop=True
            ),
        })


summary_B = pd.DataFrame([
    {
        "recording_id": r[
            "recording_id"
        ],
        "label": r[
            "label"
        ],
        "frames": len(
            r[
                "raw"
            ]
        ),
        "source_file": r[
            "source_file"
        ],
    }
    for r in recordings_B
]).sort_values(
    [
        "label",
        "recording_id",
    ]
).reset_index(
    drop=True
)

display(
    summary_B
)

class_counts_B = (
    summary_B
    .groupby(
        "label"
    )[
        "recording_id"
    ]
    .count()
)

display(
    class_counts_B
)

assert len(
    recordings_B
) == 12

assert (
    class_counts_B
    == 3
).all()

## 8. Combined final dataset

In [ ]:
recordings = (
    recordings_A
    + recordings_B
)

recording_summary = pd.DataFrame([
    {
        "recording_id": r[
            "recording_id"
        ],
        "label": r[
            "label"
        ],
        "environment": r[
            "environment"
        ],
        "frames": len(
            r[
                "raw"
            ]
        ),
        "source_file": r[
            "source_file"
        ],
    }
    for r in recordings
]).sort_values(
    [
        "environment",
        "label",
        "recording_id",
    ]
).reset_index(
    drop=True
)

display(
    recording_summary
)

print(
    "TOTAL recordings:",
    len(
        recordings
    ),
)

display(
    recording_summary.groupby(
        [
            "environment",
            "label",
        ]
    )[
        "recording_id"
    ]
    .count()
    .unstack(
        fill_value=0
    )
)

assert len(
    recordings
) == 25

assert recording_summary[
    "recording_id"
].nunique() == 25

recording_summary.to_csv(
    DATASET_CSV,
    index=False,
)

## 9. W20_HSG preprocessing

In [ ]:
def preprocess_partition(
    df
):
    processed = pd.DataFrame(
        index=df.index
    )

    for col in df.columns:
        values = df[
            col
        ].astype(
            float
        )

        h = hampel(
            values,
            window_size=HAMPEL_WINDOW,
        )

        smoothed = savgol_filter(
            np.asarray(
                h.filtered_data,
                dtype=float,
            ),
            window_length=SAVGOL_WINDOW,
            polyorder=SAVGOL_POLYORDER,
        )

        processed[
            col
        ] = smoothed

    processed.columns = range(
        processed.shape[
            1
        ]
    )

    return (
        processed
        .drop(
            columns=COLUMNS_TO_DROP,
            axis=1,
        )
        .reset_index(
            drop=True
        )
    )


def segment_processed(
    df,
    label,
    recording_id,
    environment,
):
    rows = []

    for start in range(
        0,
        len(
            df
        )
        - WINDOW_LENGTH
        + 1,
        STRIDE,
    ):
        end = (
            start
            + WINDOW_LENGTH
        )

        rows.append({
            "X": df.iloc[
                start:end
            ].values.astype(
                np.float32
            ),
            "label": label,
            "recording_id": recording_id,
            "environment": environment,
        })

    return rows

## 10. Precompute full / train80 / validation20 partitions

Filtering is performed on each complete raw partition before segmentation, matching the earlier corrected pipeline.

In [ ]:
cache_signature = {
    "recordings": sorted(
        (
            r[
                "recording_id"
            ],
            len(
                r[
                    "raw"
                ]
            ),
        )
        for r in recordings
    ),
    "window_length": WINDOW_LENGTH,
    "stride": STRIDE,
    "train_fraction": DEV_TRAIN_FRACTION,
    "columns_to_drop": COLUMNS_TO_DROP,
    "hampel_window": HAMPEL_WINDOW,
    "savgol_window": SAVGOL_WINDOW,
    "savgol_polyorder": SAVGOL_POLYORDER,
    "version": 1,
}

partition_cache = None

if CACHE_FILE.exists():
    saved = joblib.load(
        CACHE_FILE
    )

    if saved.get(
        "signature"
    ) == cache_signature:
        partition_cache = saved[
            "partition_cache"
        ]

        print(
            "Loaded preprocessing cache:",
            CACHE_FILE,
        )


if partition_cache is None:
    partition_cache = {}

    t0 = time.time()

    min_filter_frames = max(
        HAMPEL_WINDOW,
        SAVGOL_WINDOW,
    )

    for index, recording in enumerate(
        recordings,
        start=1,
    ):
        raw = recording[
            "raw"
        ].reset_index(
            drop=True
        )

        cut = int(
            len(
                raw
            )
            * DEV_TRAIN_FRACTION
        )

        train_raw = raw.iloc[
            :cut
        ].reset_index(
            drop=True
        )

        val_raw = raw.iloc[
            cut:
        ].reset_index(
            drop=True
        )

        if (
            len(
                train_raw
            )
            < min_filter_frames
            or len(
                val_raw
            )
            < min_filter_frames
        ):
            raise ValueError(
                f"{recording['recording_id']}: partition too short for filtering "
                f"(train={len(train_raw)}, val={len(val_raw)})"
            )

        full_processed = preprocess_partition(
            raw
        )

        train_processed = preprocess_partition(
            train_raw
        )

        val_processed = preprocess_partition(
            val_raw
        )

        partition_cache[
            recording[
                "recording_id"
            ]
        ] = {
            "label": recording[
                "label"
            ],
            "environment": recording[
                "environment"
            ],
            "full": segment_processed(
                full_processed,
                recording[
                    "label"
                ],
                recording[
                    "recording_id"
                ],
                recording[
                    "environment"
                ],
            ),
            "train": segment_processed(
                train_processed,
                recording[
                    "label"
                ],
                recording[
                    "recording_id"
                ],
                recording[
                    "environment"
                ],
            ),
            "val": segment_processed(
                val_processed,
                recording[
                    "label"
                ],
                recording[
                    "recording_id"
                ],
                recording[
                    "environment"
                ],
            ),
        }

        item = partition_cache[
            recording[
                "recording_id"
            ]
        ]

        print(
            f"[{index:02d}/25] "
            f"{recording['recording_id']} | "
            f"full={len(item['full'])}, "
            f"train={len(item['train'])}, "
            f"val={len(item['val'])}"
        )

    joblib.dump(
        {
            "signature": cache_signature,
            "partition_cache": partition_cache,
        },
        CACHE_FILE,
        compress=3,
    )

    print(
        f"\nPreprocessing finished in "
        f"{(time.time() - t0) / 60:.2f} min"
    )

assert len(
    partition_cache
) == 25

## 11. Label encoder and BiLSTM

In [ ]:
ALL_CLASSES = sorted(
    recording_summary[
        "label"
    ].unique()
)

label_encoder = LabelEncoder()

label_encoder.fit(
    ALL_CLASSES
)

print(
    "Class order:",
    list(
        label_encoder.classes_
    ),
)


def build_model():
    model = keras.Sequential([
        keras.Input(
            shape=(
                WINDOW_LENGTH,
                N_FEATURES,
            )
        ),
        layers.Bidirectional(
            layers.LSTM(
                64
            )
        ),
        layers.Dropout(
            0.5
        ),
        layers.Dense(
            len(
                label_encoder.classes_
            ),
            activation="softmax",
        ),
    ])

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy"
        ],
    )

    return model

## 12. Build the 25-fold leave-one-recording-out plan

There is no random test split here. Each recording is held out exactly once.

In [ ]:
recording_table = recording_summary[
    [
        "recording_id",
        "label",
        "environment",
    ]
].copy().reset_index(
    drop=True
)

fold_plan = []

for test_index in range(
    len(
        recording_table
    )
):
    test_id = recording_table.loc[
        test_index,
        "recording_id",
    ]

    development_ids = recording_table.loc[
        recording_table.index
        != test_index,
        "recording_id",
    ].tolist()

    fold_plan.append({
        "fold": test_index + 1,
        "development_ids": development_ids,
        "test_id": test_id,
    })


assert len(
    fold_plan
) == 25

all_test_ids = [
    fold[
        "test_id"
    ]
    for fold in fold_plan
]

assert len(
    set(
        all_test_ids
    )
) == 25

assert set(
    all_test_ids
) == set(
    recording_table[
        "recording_id"
    ]
)

plan_df = pd.DataFrame([
    {
        "fold": fold[
            "fold"
        ],
        "test_recording": fold[
            "test_id"
        ],
        "test_label": recording_table.loc[
            recording_table[
                "recording_id"
            ]
            == fold[
                "test_id"
            ],
            "label",
        ].iloc[
            0
        ],
        "test_environment": recording_table.loc[
            recording_table[
                "recording_id"
            ]
            == fold[
                "test_id"
            ],
            "environment",
        ].iloc[
            0
        ],
        "development_recordings": len(
            fold[
                "development_ids"
            ]
        ),
    }
    for fold in fold_plan
])

display(
    plan_df
)

print(
    "Every one of the 25 recordings is TEST exactly once — PASSED"
)

## 13. Helpers for training and checkpointing

In [ ]:
def collect_rows(
    rows
):
    X = np.stack(
        [
            row[
                "X"
            ]
            for row in rows
        ]
    ).astype(
        np.float32
    )

    y = label_encoder.transform(
        [
            row[
                "label"
            ]
            for row in rows
        ]
    ).astype(
        np.int64
    )

    return (
        X,
        y,
    )


def apply_scaler(
    scaler,
    X,
):
    z = scaler.transform(
        X.reshape(
            len(
                X
            ),
            -1,
        )
    )

    return z.reshape(
        len(
            X
        ),
        WINDOW_LENGTH,
        N_FEATURES,
    ).astype(
        np.float32
    )


def fold_checkpoint_paths(
    fold_index
):
    base = (
        f"fold_{fold_index:02d}"
    )

    return {
        "metrics": CHECKPOINT_DIR / f"{base}_metrics.json",
        "predictions": CHECKPOINT_DIR / f"{base}_predictions.csv",
        "history": CHECKPOINT_DIR / f"{base}_history.joblib",
    }


def fold_is_complete(
    fold_index
):
    paths = fold_checkpoint_paths(
        fold_index
    )

    return all(
        path.exists()
        for path in paths.values()
    )

## 14. Run the 25 LORO folds

Completed folds are automatically skipped if their checkpoint files already exist.

In [ ]:
fold_metrics = []
prediction_tables = []
histories = {}

experiment_start = time.time()

for fold in fold_plan:
    fold_index = fold[
        "fold"
    ]

    development_ids = fold[
        "development_ids"
    ]

    test_id = fold[
        "test_id"
    ]

    checkpoint_paths = fold_checkpoint_paths(
        fold_index
    )

    if fold_is_complete(
        fold_index
    ):
        with checkpoint_paths[
            "metrics"
        ].open(
            "r",
            encoding="utf-8",
        ) as fh:
            fold_metric = json.load(
                fh
            )

        fold_metrics.append(
            fold_metric
        )

        prediction_tables.append(
            pd.read_csv(
                checkpoint_paths[
                    "predictions"
                ]
            )
        )

        histories[
            fold_index
        ] = joblib.load(
            checkpoint_paths[
                "history"
            ]
        )

        print(
            f"FOLD {fold_index:02d}/25 already completed — loaded checkpoint"
        )

        continue

    test_info = recording_table[
        recording_table[
            "recording_id"
        ]
        == test_id
    ].iloc[
        0
    ]

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"FOLD {fold_index:02d}/25 | "
        f"TEST={test_id} | "
        f"{test_info['label']} | "
        f"{test_info['environment']}"
    )

    print(
        "=" * 90
    )

    train_rows = []
    val_rows = []

    for recording_id in development_ids:
        train_rows.extend(
            partition_cache[
                recording_id
            ][
                "train"
            ]
        )

        val_rows.extend(
            partition_cache[
                recording_id
            ][
                "val"
            ]
        )

    test_rows = partition_cache[
        test_id
    ][
        "full"
    ]

    train_val_ids = {
        row[
            "recording_id"
        ]
        for row in (
            train_rows
            + val_rows
        )
    }

    assert test_id not in train_val_ids

    if len(
        val_rows
    ) == 0:
        raise ValueError(
            f"Fold {fold_index}: no validation windows."
        )

    X_train_raw, y_train = collect_rows(
        train_rows
    )

    X_val_raw, y_val = collect_rows(
        val_rows
    )

    X_test_raw, y_test = collect_rows(
        test_rows
    )

    scaler = StandardScaler()

    scaler.fit(
        X_train_raw.reshape(
            len(
                X_train_raw
            ),
            -1,
        )
    )

    X_train = apply_scaler(
        scaler,
        X_train_raw,
    )

    X_val = apply_scaler(
        scaler,
        X_val_raw,
    )

    X_test = apply_scaler(
        scaler,
        X_test_raw,
    )

    keras.backend.clear_session()

    random.seed(
        SEED
    )

    np.random.seed(
        SEED
    )

    tf.keras.utils.set_random_seed(
        SEED
    )

    model = build_model()

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
    )

    fold_start = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(
            X_val,
            y_val,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            early_stopping
        ],
        shuffle=True,
        verbose=1,
    )

    probabilities = model.predict(
        X_test,
        verbose=0,
    )

    y_pred = np.argmax(
        probabilities,
        axis=1,
    )

    accuracy = accuracy_score(
        y_test,
        y_pred,
    )

    best_epoch = int(
        np.argmin(
            history.history[
                "val_loss"
            ]
        )
        + 1
    )

    fold_minutes = (
        time.time()
        - fold_start
    ) / 60

    fold_metric = {
        "fold": int(
            fold_index
        ),
        "test_recording": str(
            test_id
        ),
        "test_label": str(
            test_info[
                "label"
            ]
        ),
        "test_environment": str(
            test_info[
                "environment"
            ]
        ),
        "train_windows": int(
            len(
                y_train
            )
        ),
        "validation_windows": int(
            len(
                y_val
            )
        ),
        "test_windows": int(
            len(
                y_test
            )
        ),
        "epochs_run": int(
            len(
                history.history[
                    "loss"
                ]
            )
        ),
        "best_epoch": int(
            best_epoch
        ),
        "accuracy": float(
            accuracy
        ),
        "fold_minutes": float(
            fold_minutes
        ),
    }

    history_dict = {
        key: [
            float(
                value
            )
            for value in values
        ]
        for key, values in history.history.items()
    }

    prediction_table = pd.DataFrame({
        "fold": fold_index,
        "recording_id": test_id,
        "environment": test_info[
            "environment"
        ],
        "true_label": label_encoder.inverse_transform(
            y_test
        ),
        "predicted_label": label_encoder.inverse_transform(
            y_pred
        ),
        "confidence": np.max(
            probabilities,
            axis=1,
        ),
    })

    with checkpoint_paths[
        "metrics"
    ].open(
        "w",
        encoding="utf-8",
    ) as fh:
        json.dump(
            fold_metric,
            fh,
            indent=2,
        )

    prediction_table.to_csv(
        checkpoint_paths[
            "predictions"
        ],
        index=False,
    )

    joblib.dump(
        history_dict,
        checkpoint_paths[
            "history"
        ],
        compress=3,
    )

    fold_metrics.append(
        fold_metric
    )

    prediction_tables.append(
        prediction_table
    )

    histories[
        fold_index
    ] = history_dict

    completed_count = len(
        fold_metrics
    )

    elapsed = (
        time.time()
        - experiment_start
    ) / 60

    newly_run = sum(
        not fold_is_complete(
            f[
                "fold"
            ]
        )
        for f in []
    )

    print(
        f"Accuracy={accuracy:.4f} | "
        f"best_epoch={best_epoch} | "
        f"fold={fold_minutes:.1f} min"
    )


fold_metrics_df = pd.DataFrame(
    fold_metrics
).sort_values(
    "fold"
).reset_index(
    drop=True
)

all_predictions = pd.concat(
    prediction_tables,
    ignore_index=True,
).sort_values(
    [
        "fold",
        "recording_id",
    ]
).reset_index(
    drop=True
)

assert len(
    fold_metrics_df
) == 25

assert all_predictions[
    "recording_id"
].nunique() == 25

display(
    fold_metrics_df
)

fold_metrics_df.to_csv(
    FOLD_CSV,
    index=False,
)

all_predictions.to_csv(
    PREDICTIONS_CSV,
    index=False,
)

## 15. Main aggregated 25-fold LORO result

This is the principal offline generalization result.

In [ ]:
y_true = all_predictions[
    "true_label"
].to_numpy()

y_pred = all_predictions[
    "predicted_label"
].to_numpy()

overall_accuracy = accuracy_score(
    y_true,
    y_pred,
)

overall_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

overall_weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0,
)

print(
    "=== FINAL A13 + B12 | W20_HSG | 25-FOLD LORO ==="
)

print(
    f"Aggregated Accuracy:    {overall_accuracy:.4f}"
)

print(
    f"Aggregated Macro F1:    {overall_macro_f1:.4f}"
)

print(
    f"Aggregated Weighted F1: {overall_weighted_f1:.4f}"
)

print()

print(
    "Per-recording accuracy mean ± SD:"
)

print(
    f"{fold_metrics_df['accuracy'].mean():.4f} "
    f"± "
    f"{fold_metrics_df['accuracy'].std(ddof=1):.4f}"
)

print(
    f"Median per-recording accuracy: "
    f"{fold_metrics_df['accuracy'].median():.4f}"
)

## 16. Per-class classification report

In [ ]:
report_text = classification_report(
    y_true,
    y_pred,
    labels=label_encoder.classes_,
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0,
)

print(
    report_text
)

report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=label_encoder.classes_,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
).T

display(
    report_df
)

report_df.to_csv(
    REPORT_CSV
)

## 17. Environment A and Environment B separately

In [ ]:
environment_rows = []

for environment in [
    "Environment A",
    "Environment B",
]:
    subset = all_predictions[
        all_predictions[
            "environment"
        ]
        == environment
    ]

    env_true = subset[
        "true_label"
    ].to_numpy()

    env_pred = subset[
        "predicted_label"
    ].to_numpy()

    environment_rows.append({
        "environment": environment,
        "held_out_recordings": subset[
            "recording_id"
        ].nunique(),
        "windows": len(
            subset
        ),
        "accuracy": accuracy_score(
            env_true,
            env_pred,
        ),
        "macro_f1": f1_score(
            env_true,
            env_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            env_true,
            env_pred,
            average="weighted",
            zero_division=0,
        ),
    })


environment_df = pd.DataFrame(
    environment_rows
)

display(
    environment_df
)

environment_df.to_csv(
    ENVIRONMENT_CSV,
    index=False,
)

## 18. Per-recording held-out results

In [ ]:
per_recording_df = (
    fold_metrics_df[
        [
            "fold",
            "test_recording",
            "test_label",
            "test_environment",
            "test_windows",
            "accuracy",
            "best_epoch",
            "fold_minutes",
        ]
    ]
    .rename(
        columns={
            "test_recording": "recording_id",
            "test_label": "true_activity",
            "test_environment": "environment",
            "test_windows": "windows",
        }
    )
    .sort_values(
        "accuracy"
    )
    .reset_index(
        drop=True
    )
)

display(
    per_recording_df
)

per_recording_df.to_csv(
    PER_RECORDING_CSV,
    index=False,
)

## 19. Normalized confusion matrix

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=label_encoder.classes_,
    normalize="true",
)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
)

display(
    cm_df
)

fig, ax = plt.subplots(
    figsize=(
        7,
        6,
    )
)

image = ax.imshow(
    cm,
    vmin=0,
    vmax=1,
)

ax.set_xticks(
    range(
        len(
            label_encoder.classes_
        )
    )
)

ax.set_xticklabels(
    label_encoder.classes_,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    range(
        len(
            label_encoder.classes_
        )
    )
)

ax.set_yticklabels(
    label_encoder.classes_
)

ax.set_xlabel(
    "Predicted class"
)

ax.set_ylabel(
    "True class"
)

ax.set_title(
    "A13 + B12 | W20_HSG | 25-fold LORO"
)

for i in range(
    cm.shape[
        0
    ]
):
    for j in range(
        cm.shape[
            1
        ]
    ):
        ax.text(
            j,
            i,
            f"{cm[i, j]:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(
    image,
    ax=ax,
    label="Proportion",
)

fig.tight_layout()

fig.savefig(
    CONFUSION_PDF,
    bbox_inches="tight",
)

plt.show()

## 20. Per-recording accuracy plot

In [ ]:
plot_df = per_recording_df.sort_values(
    "accuracy",
    ascending=True,
)

fig, ax = plt.subplots(
    figsize=(
        10,
        8,
    )
)

positions = np.arange(
    len(
        plot_df
    )
)

ax.barh(
    positions,
    plot_df[
        "accuracy"
    ],
)

ax.set_yticks(
    positions
)

ax.set_yticklabels(
    plot_df[
        "recording_id"
    ]
)

ax.axvline(
    overall_accuracy,
    linestyle="--",
    label=(
        f"Aggregated accuracy = "
        f"{overall_accuracy:.3f}"
    ),
)

ax.set_xlim(
    0,
    1,
)

ax.set_xlabel(
    "Held-out recording accuracy"
)

ax.set_ylabel(
    "Recording"
)

ax.set_title(
    "W20_HSG — 25-fold LORO per-recording accuracy"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    PER_RECORDING_PDF,
    bbox_inches="tight",
)

plt.show()

## 21. Mean learning curves across 25 folds

Because early stopping gives different fold lengths, curves are averaged over the common epoch range.

In [ ]:
minimum_epochs = min(
    len(
        history[
            "loss"
        ]
    )
    for history in histories.values()
)

epochs = np.arange(
    1,
    minimum_epochs
    + 1,
)


def history_matrix(
    metric
):
    return np.asarray([
        histories[
            fold_index
        ][
            metric
        ][
            :minimum_epochs
        ]
        for fold_index in sorted(
            histories
        )
    ], dtype=float)


train_loss = history_matrix(
    "loss"
)

val_loss = history_matrix(
    "val_loss"
)

train_accuracy = history_matrix(
    "accuracy"
)

val_accuracy = history_matrix(
    "val_accuracy"
)


fig, ax = plt.subplots(
    figsize=(
        8,
        5,
    )
)

ax.plot(
    epochs,
    train_loss.mean(
        axis=0
    ),
    label="Training loss",
)

ax.plot(
    epochs,
    val_loss.mean(
        axis=0
    ),
    label="Validation loss",
)

ax.fill_between(
    epochs,
    train_loss.mean(
        axis=0
    )
    - train_loss.std(
        axis=0
    ),
    train_loss.mean(
        axis=0
    )
    + train_loss.std(
        axis=0
    ),
    alpha=0.15,
)

ax.fill_between(
    epochs,
    val_loss.mean(
        axis=0
    )
    - val_loss.std(
        axis=0
    ),
    val_loss.mean(
        axis=0
    )
    + val_loss.std(
        axis=0
    ),
    alpha=0.15,
)

ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Loss"
)

ax.set_title(
    "W20_HSG — 25-fold LORO mean learning loss"
)

ax.grid(
    True,
    alpha=0.3,
)

ax.legend()

fig.tight_layout()

fig.savefig(
    LOSS_PDF,
    bbox_inches="tight",
)

plt.show()


fig, ax = plt.subplots(
    figsize=(
        8,
        5,
    )
)

ax.plot(
    epochs,
    train_accuracy.mean(
        axis=0
    ),
    label="Training accuracy",
)

ax.plot(
    epochs,
    val_accuracy.mean(
        axis=0
    ),
    label="Validation accuracy",
)

ax.fill_between(
    epochs,
    train_accuracy.mean(
        axis=0
    )
    - train_accuracy.std(
        axis=0
    ),
    train_accuracy.mean(
        axis=0
    )
    + train_accuracy.std(
        axis=0
    ),
    alpha=0.15,
)

ax.fill_between(
    epochs,
    val_accuracy.mean(
        axis=0
    )
    - val_accuracy.std(
        axis=0
    ),
    val_accuracy.mean(
        axis=0
    )
    + val_accuracy.std(
        axis=0
    ),
    alpha=0.15,
)

ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Accuracy"
)

ax.set_ylim(
    0,
    1.02,
)

ax.set_title(
    "W20_HSG — 25-fold LORO mean learning accuracy"
)

ax.grid(
    True,
    alpha=0.3,
)

ax.legend()

fig.tight_layout()

fig.savefig(
    ACCURACY_PDF,
    bbox_inches="tight",
)

plt.show()

## 22. Compact final summary

In [ ]:
final_summary = pd.DataFrame([
    {
        "protocol": "25-fold leave-one-recording-out",
        "recordings": 25,
        "environment_A_recordings": 13,
        "environment_B_recordings": 12,
        "window": WINDOW_LENGTH,
        "hampel": True,
        "savgol": True,
        "accuracy": overall_accuracy,
        "macro_f1": overall_macro_f1,
        "weighted_f1": overall_weighted_f1,
        "per_recording_accuracy_mean": fold_metrics_df[
            "accuracy"
        ].mean(),
        "per_recording_accuracy_std": fold_metrics_df[
            "accuracy"
        ].std(
            ddof=1
        ),
        "per_recording_accuracy_median": fold_metrics_df[
            "accuracy"
        ].median(),
    }
])

display(
    final_summary
)

## 23. Package all outputs

In [ ]:
result_files = [
    DATASET_CSV,
    FOLD_CSV,
    PREDICTIONS_CSV,
    REPORT_CSV,
    ENVIRONMENT_CSV,
    PER_RECORDING_CSV,
    CONFUSION_PDF,
    PER_RECORDING_PDF,
    LOSS_PDF,
    ACCURACY_PDF,
]

with zipfile.ZipFile(
    RESULT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file_path in result_files:
        if file_path.exists():
            zf.write(
                file_path,
                arcname=file_path.name,
            )

print(
    "Created:",
    RESULT_ZIP,
)

try:
    from google.colab import files

    files.download(
        str(
            RESULT_ZIP
        )
    )

except ImportError:
    pass